# H03 — Time series: lecture 9 in examples

**What this is.** Eight short examples of lecture 9, run in class and four short exercises for you.
- **The examples** have one line `# change these`. Change a value, run the cell again and look at the figure.
- **The exercises** have an empty cell: write a few lines, compare with *What you should see*, then open the solution.

**The conventions of lecture 9.**
- The time step is one sample, so $f$ is in cycles per sample, from $-1/2$ to $1/2$.
- $r_{xx}[k]$ is the autocorrelation at the lag $k$.
- $S_{xx}(f)$ is the power spectral density, two-sided, per cycle per sample.

| Example | What we see |
|---|---|
| 1 | White and colored noise in time |
| 2 | Their autocorrelation |
| 3 | Their power spectral density |
| 4 | Two memory terms: a peak in the PSD |
| 5 | One periodogram: its scatter does not go down with $N$ |
| 6 | Averaging $K$ segments: less scatter, coarser resolution |
| 7 | The window: the rectangle and Hann, in time and in frequency |
| 8 | Bias and variance: the length of a segment |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, os

# the course root holds gwcourse_palette.py; find it from wherever we are
here = os.path.abspath('.')
while not os.path.exists(os.path.join(here, 'gwcourse_palette.py')):
    if os.path.dirname(here) == here:
        raise FileNotFoundError('gwcourse_palette.py not found: open the notebook from the course folder')
    here = os.path.dirname(here)
sys.path.insert(0, here)

from gwcourse_palette import P, apply_style
apply_style()

In [ ]:
def colored_noise(a, N, sigma_w, seed):
    """Return white noise w[n] and the colored noise x[n] = a x[n-1] + w[n], N samples each.

    The first 1000 samples of x are dropped, so the start does not matter."""
    rng = np.random.default_rng(seed)
    w = sigma_w * rng.normal(size=N + 1000)
    # the filter of lecture 9, one sample at a time
    x = np.zeros_like(w)
    for n in range(1, len(w)):
        x[n] = a * x[n - 1] + w[n]
    w_kept = w[1000:]
    x_kept = x[1000:]
    return w_kept, x_kept


def acf_estimate(x, k_max):
    """Return r_xx[k] for k = 0 ... k_max, as the time average of x[n] x[n+k]."""
    N = len(x)
    r = np.array([np.mean(x[:N - k] * x[k:]) for k in range(k_max + 1)])
    return r


def periodogram(x):
    """Return the frequencies 0 ... 1/2 and the periodogram (1/N) |sum x[n] e^(-i 2 pi f n)|^2."""
    N = len(x)
    X = np.fft.rfft(x)
    f = np.fft.rfftfreq(N)
    S_hat = np.abs(X) ** 2 / N
    return f, S_hat


def exact_psd(f, a, sigma_w):
    """Return S_xx(f) = sigma_w^2 / |1 - a e^(-i 2 pi f)|^2, the PSD of lecture 9."""
    S = sigma_w ** 2 / np.abs(1 - a * np.exp(-2j * np.pi * f)) ** 2
    return S

## 1. White and colored noise

**White noise** $w[n]$: every sample is new. **Colored noise** $x[n] = a\,x[n-1] + w[n]$: each sample keeps a fraction $a$ of the one before.

**Look at:** the white record jumps; the colored one wanders. Try $a = 0.5$, $0.9$, $0.99$.

In [ ]:
# change these
a, sigma_w, N = 0.9, 1.0, 4096

w, x = colored_noise(a, N, sigma_w, seed=1)
n = np.arange(300)
fig, ax = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True)
ax[0].plot(n, w[:300], color=P['GREY_DARK'], lw=0.9)
ax[0].set_ylabel('white $w[n]$')
ax[1].plot(n, x[:300], color=P['BLUE'], lw=0.9)
ax[1].set(xlabel='sample $n$', ylabel=f'colored $x[n]$, $a$ = {a}')
plt.tight_layout()
plt.show()

## 2. The autocorrelation

**The formulas of lecture 9.**
- White: $r_{ww}[k] = \sigma_{w}^{2}\,\delta[k]$, a single spike at $k = 0$.
- Colored: $r_{xx}[k] = \sigma_{w}^{2}\,a^{|k|}/(1 - a^{2})$, a memory that decays.

**Look at:** the estimate (dots) against the formula (line). Change $a$: the memory gets longer as $a \to 1$. Change $N$: the estimate gets closer to the formula.

In [ ]:
# change these
a, sigma_w, N, k_max = 0.9, 1.0, 4096, 40

w, x = colored_noise(a, N, sigma_w, seed=1)
k = np.arange(k_max + 1)
r_w = acf_estimate(w, k_max)
r_x = acf_estimate(x, k_max)
r_x_exact = sigma_w ** 2 * a ** k / (1 - a ** 2)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].stem(k, r_w, linefmt=P['GREY_MID'], markerfmt='o', basefmt=' ')
ax[0].set(xlabel='lag $k$', ylabel='$r_{ww}[k]$', title='white')
ax[1].plot(k, r_x, 'o', color=P['BLUE'], ms=4, label='estimate')
ax[1].plot(k, r_x_exact, color=P['BRICK'], lw=1.6, label='formula')
ax[1].set(xlabel='lag $k$', ylabel='$r_{xx}[k]$', title=f'colored, $a$ = {a}')
ax[1].legend(frameon=False)
plt.tight_layout()
plt.show()

### 2b. The autocorrelation with NumPy and SciPy

**The question.** NumPy and SciPy have functions for the correlation. Do they give our estimate?

**The three estimates.**
- **By hand**, `acf_estimate`: the time average of $x[n]\,x[n+k]$. For lag $k$ there are $N - k$ products, so we divide by $N - k$.
- **NumPy**, `np.correlate(x, x, mode='full')`: the sum $\sum_{n} x[n]\,x[n+k]$ for every lag, from $-(N-1)$ to $N-1$. It does not divide. We keep the lags $k \ge 0$. It computes the sum directly, so its time grows as $N^{2}$: on long records it is very slow. The cell uses it only for $N \le 8192$.
- **SciPy**, `scipy.signal.correlate(x, x, method='fft')`: the same sum, computed with the FFT. Its time grows as $N \log N$: it is fast for every $N$.

**Two normalizations.**
- Divide the sum by $N - k$: the **unbiased** estimate, equal to ours.
- Divide the sum by $N$: the **biased** estimate. For small $k$ the two are almost equal. For $k$ close to $N$ the biased one goes to zero.

**Look at:** the three estimates agree. The cell prints the largest difference and the time of each function. Change $a$ and $N$; for $N > 8192$ only SciPy is used.

In [ ]:
from scipy import signal
import time

# change these
a, sigma_w, N, k_max = 0.9, 1.0, 4096, 40

w, x = colored_noise(a, N, sigma_w, seed=1)
k = np.arange(k_max + 1)

# 1. by hand: one lag at a time
t0 = time.perf_counter()
r_hand = acf_estimate(x, k_max)
t_hand = time.perf_counter() - t0

# 2. SciPy, with the FFT: the sum for every lag; the lag 0 is at index N - 1
t0 = time.perf_counter()
c_scipy = signal.correlate(x, x, mode='full', method='fft')
t_scipy = time.perf_counter() - t0
sum_scipy = c_scipy[N - 1:N + k_max]

# 3. NumPy: the same sum, computed directly; only for short records
if N <= 8192:
    t0 = time.perf_counter()
    c_numpy = np.correlate(x, x, mode='full')
    t_numpy = time.perf_counter() - t0
    sum_numpy = c_numpy[N - 1:N + k_max]
else:
    print("N > 8192: np.correlate would take minutes, we skip it")
    sum_numpy = sum_scipy
    t_numpy = float('nan')

# the two normalizations
r_unbiased = sum_scipy / (N - k)
r_biased = sum_scipy / N

print(f"largest difference, hand - NumPy (unbiased):  {np.max(np.abs(r_hand - sum_numpy / (N - k))):.1e}")
print(f"largest difference, NumPy - SciPy:            {np.max(np.abs(sum_numpy - sum_scipy)):.1e}")
print(f"time: by hand {1e3 * t_hand:.1f} ms (only {k_max + 1} lags), "
      f"NumPy {1e3 * t_numpy:.1f} ms, SciPy FFT {1e3 * t_scipy:.1f} ms (all {2 * N - 1} lags)")

r_exact = sigma_w ** 2 * a ** k / (1 - a ** 2)
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(k, r_exact, color=P['BRICK'], lw=1.8, label='formula')
ax.plot(k, r_hand, 'o', color=P['BLUE'], ms=6, label='by hand')
ax.plot(k, r_unbiased, 'x', color=P['AMBER_DARK'], ms=7, label='SciPy, divided by $N-k$')
ax.plot(k, r_biased, '.', color=P['GREY_DARK'], ms=5, label='SciPy, divided by $N$')
ax.set(xlabel='lag $k$', ylabel='$r_{xx}[k]$', title=f'colored noise, $a$ = {a}, $N$ = {N}')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

### Exercise 1 · The autocorrelation of white noise

Make $N = 4096$ samples of white noise with $\sigma_{w} = 1$: `w = np.random.default_rng(5).normal(size=N)`. Estimate $r_{ww}[k]$ for $k = 0, 1, \dots, 20$ with a loop over $k$ and `np.mean(w[:N - k] * w[k:])`. Print $r_{ww}[0]$, the largest $|r_{ww}[k]|$ for $k \ge 1$ and how many values lie inside $\pm 2/\sqrt{N}$.

**What you should see.** $r_{ww}[0]$ near 1, the variance. The other values are small, almost all inside $\pm 2/\sqrt{N} = 0.031$: white noise has no memory.

In [ ]:
# your code here

<details>
<summary>Solution</summary>

```python
N = 4096
w = np.random.default_rng(5).normal(size=N)
# the time average of w[n] w[n+k], one lag at a time
r_w = np.array([np.mean(w[:N - k] * w[k:]) for k in range(21)])
band = 2 / np.sqrt(N)
inside = np.sum(np.abs(r_w[1:]) < band)
print(f"r_ww[0] = {r_w[0]:.3f}   largest |r_ww[k]| for k >= 1: {np.abs(r_w[1:]).max():.3f}")
print(f"{inside} of 20 values inside +-{band:.3f}")
```
$r_{ww}[0] = 0.997$, the variance $\sigma_{w}^{2} = 1$ within the scatter. For $k \ge 1$ the largest value is $0.041$ and 18 of the 20 values lie inside $\pm 0.031$. The exact ACF is $\sigma_{w}^{2}\delta[k]$: what we see at $k \ne 0$ is the error of the estimate, about $1/\sqrt{N}$.

</details>


## 3. The power spectral density

**The formulas of lecture 9.**
- White: $S_{ww}(f) = \sigma_{w}^{2}$, flat.
- Colored: $S_{xx}(f) = \sigma_{w}^{2}/|1 - a\,\mathrm{e}^{-\mathrm{i} 2\pi f}|^{2}$, the power at low $f$.
- The area under $S_{xx}$ from $-1/2$ to $1/2$ is the variance $r_{xx}[0]$.

**Look at:** the memory in time is the power at low frequency. Change $a$: a longer memory gives a narrower and taller peak at $f = 0$. The cell prints the area against the variance.

In [ ]:
# change these
a, sigma_w = 0.9, 1.0

f = np.linspace(-0.5, 0.5, 2001)
S_x = exact_psd(f, a, sigma_w)
area = np.trapezoid(S_x, f)
print(f"area under S_xx = {area:.3f}   variance sigma_w^2/(1 - a^2) = {sigma_w**2 / (1 - a**2):.3f}")

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.semilogy(f, S_x, color=P['BLUE'], lw=1.8, label=f'colored, $a$ = {a}')
ax.axhline(sigma_w ** 2, color=P['GREY_DARK'], lw=1.4, ls='--', label='white')
ax.set(xlabel='$f$ (cycles per sample)', ylabel='$S(f)$', xlim=(-0.5, 0.5))
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## 4. Two memory terms: a peak

**A filter with two memory terms.** $x[n] = a_{1}x[n-1] + a_{2}x[n-2] + w[n]$. We choose the two coefficients from three numbers we can read on the plot:
- $f_{r}$, the frequency of the peak, in cycles per sample;
- $r$, between 0 and 1: the closer to 1, the taller and narrower the peak;
- $\sigma_{w}$, the rms of the white noise, which sets the level.

The coefficients are $a_{1} = 2r\cos(2\pi f_{r})$ and $a_{2} = -r^{2}$. The PSD is

$$
S_{xx}(f) = \frac{\sigma_{w}^{2}}{\left|1 - a_{1}\mathrm{e}^{-\mathrm{i} 2\pi f} - a_{2}\mathrm{e}^{-\mathrm{i} 4\pi f}\right|^{2}} .
$$

This is the noise of a resonance, like the mirror on its spring.

**Look at:** the record rings near $f_{r}$. Move $f_{r}$ (0.05, 0.1, 0.25): the peak moves. Take $r$ = 0.8, 0.95, 0.99: the peak becomes taller and narrower and the record rings longer. Change $\sigma_{w}$: the whole curve moves up or down.

In [ ]:
# change these
f_r, r, sigma_w = 0.1, 0.95, 1.0

# the two coefficients from the peak frequency and the pole radius
a1 = 2 * r * np.cos(2 * np.pi * f_r)
a2 = -r ** 2

# the record: two memory terms, one sample at a time
N = 16384
w = sigma_w * np.random.default_rng(9).normal(size=N + 1000)
x = np.zeros_like(w)
for n in range(2, len(w)):
    x[n] = a1 * x[n - 1] + a2 * x[n - 2] + w[n]
x = x[1000:]

# the exact PSD and the average of K = 16 periodograms
f_exact = np.linspace(0, 0.5, 2001)
S_exact = sigma_w ** 2 / np.abs(1 - a1 * np.exp(-2j * np.pi * f_exact) - a2 * np.exp(-4j * np.pi * f_exact)) ** 2
K = 16
D = N // K
S_avg = np.mean([periodogram(seg)[1] for seg in x.reshape(K, D)], axis=0)
f_avg = periodogram(x[:D])[0]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={'width_ratios': [1, 1.2]})
ax[0].plot(np.arange(200), x[:200], color=P['BLUE'], lw=0.9)
ax[0].set(xlabel='sample $n$', ylabel='$x[n]$', title=f'$f_r$ = {f_r}, $r$ = {r}')
ax[1].semilogy(f_avg, S_avg, color=P['GREY_MID'], lw=0.8, label='average of 16 periodograms')
ax[1].semilogy(f_exact, S_exact, color=P['BRICK'], lw=1.8, label='exact $S_{xx}$')
ax[1].set(xlabel='$f$ (cycles per sample)', ylabel='$S(f)$', xlim=(0, 0.5))
ax[1].legend(frameon=False, loc='upper right')
plt.tight_layout()
plt.show()

## 5. One periodogram

**The periodogram** of one record of $N$ samples is $\hat{S}(f) = \frac{1}{N}\left|\sum_{n} x[n]\,\mathrm{e}^{-\mathrm{i} 2\pi f n}\right|^{2}$.

**Look at:** it scatters around the exact PSD by as much as the PSD itself. Change $N$ from 512 to 32768: the points get closer together in $f$, but the scatter does not go down. The cell prints the scatter of $\hat{S}/S_{xx}$.

In [ ]:
# change these
a, sigma_w, N = 0.9, 1.0, 4096

w, x = colored_noise(a, N, sigma_w, seed=2)
f, S_hat = periodogram(x)
S_x = exact_psd(f, a, sigma_w)
u = S_hat[1:-1] / S_x[1:-1]
print(f"N = {N}: {len(f)} frequencies, scatter of S_hat/S_xx = {u.std():.2f} (mean {u.mean():.2f})")

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.semilogy(f, S_hat, color=P['GREY_MID'], lw=0.6, label='one periodogram')
ax.semilogy(f, S_x, color=P['BRICK'], lw=1.8, label='exact $S_{xx}$')
ax.set(xlabel='$f$ (cycles per sample)', ylabel='$\\hat{S}(f)$', xlim=(0, 0.5))
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

### Exercise 2 · The periodogram of white noise

Compute the periodogram of white noise with $\sigma_{w} = 1$, $\hat{S}(f) = |X(f)|^{2}/N$, with `X = np.fft.rfft(w)`. Leave out the first and the last value ($f = 0$ and $f = 1/2$). Print the mean and the standard deviation of $\hat{S}$, for $N = 1024$ and for $N = 16384$ (seed 6).

**What you should see.** The mean is near $\sigma_{w}^{2} = 1$, the flat PSD. The standard deviation is also near 1, for both $N$: a longer record does not reduce the scatter of one periodogram.

In [ ]:
# your code here

<details>
<summary>Solution</summary>

```python
for N in (1024, 16384):
    w = np.random.default_rng(6).normal(size=N)
    # the squared transform divided by N, without f = 0 and f = 1/2
    S_hat = np.abs(np.fft.rfft(w)) ** 2 / N
    S_inner = S_hat[1:-1]
    print(f"N = {N:5d}: mean {S_inner.mean():.3f}   standard deviation {S_inner.std():.3f}")
```
For $N = 1024$ the mean is $0.985$ and the standard deviation $0.991$. For $N = 16384$ they are $1.015$ and $1.016$. The mean is the PSD $\sigma_{w}^{2} = 1$. The standard deviation equals the mean at every $N$: more samples give more frequencies, not a better estimate at each one.

</details>


## 6. Averaging $K$ segments

**Welch's method.** Cut the record into $K$ segments of $D$ samples, take the periodogram of each and average. The scatter falls as $1/\sqrt{K}$. Each segment is $K$ times shorter, so the resolution $\delta f = 1/D$ is $K$ times coarser.

**Look at:** change $K$ (1, 8, 64). The curve gets smoother and the points are farther apart in $f$. The cell prints the scatter against $1/\sqrt{K}$.

In [ ]:
# change these
a, sigma_w, N, K = 0.9, 1.0, 16384, 8

w, x = colored_noise(a, N, sigma_w, seed=3)
D = N // K
segments = x[:K * D].reshape(K, D)
S_segments = [periodogram(seg)[1] for seg in segments]
f = periodogram(segments[0])[0]
S_avg = np.mean(S_segments, axis=0)
S_x = exact_psd(f, a, sigma_w)
u = S_avg[1:-1] / S_x[1:-1]
print(f"K = {K}, D = {D}, resolution 1/D = {1 / D:.5f}: scatter = {u.std():.2f}, 1/sqrt(K) = {1 / np.sqrt(K):.2f}")

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.semilogy(f, S_avg, color=P['BLUE'], lw=0.9, label=f'average of $K$ = {K}')
ax.semilogy(f, S_x, color=P['BRICK'], lw=1.8, label='exact $S_{xx}$')
ax.set(xlabel='$f$ (cycles per sample)', ylabel='$\\hat{S}(f)$', xlim=(0, 0.5))
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

### Exercise 3 · Averaging by hand

Take $N = 16384$ samples of white noise (seed 7). Cut them into $K$ segments of $D = N/K$ samples with `w.reshape(K, D)`, compute the periodogram of each segment and average them. Print the scatter (standard deviation divided by mean, without $f = 0$ and $f = 1/2$) for $K = 4$, $16$, $64$, against $1/\sqrt{K}$.

**What you should see.** The scatter falls as $1/\sqrt{K}$: about 0.5, 0.25 and 0.12. The resolution $\delta f = 1/D$ grows with $K$.

In [ ]:
# your code here

<details>
<summary>Solution</summary>

```python
N = 16384
w = np.random.default_rng(7).normal(size=N)
for K in (4, 16, 64):
    D = N // K
    segments = w.reshape(K, D)
    # the periodogram of each segment, then the average over the K segments
    S_segments = [np.abs(np.fft.rfft(seg)) ** 2 / D for seg in segments]
    S_avg = np.mean(S_segments, axis=0)[1:-1]
    scatter = S_avg.std() / S_avg.mean()
    print(f"K = {K:2d}, D = {D:5d}, resolution {1 / D:.5f}: scatter {scatter:.3f}   1/sqrt(K) = {1 / np.sqrt(K):.3f}")
```
The scatter is $0.499$, $0.246$ and $0.122$ against $1/\sqrt{K} = 0.5$, $0.25$ and $0.125$. The resolution is $1/D = 0.00024$, $0.00098$ and $0.0039$ cycles per sample. For white noise the coarser resolution costs nothing, because there is no feature to smooth. Example 6 shows the cost on colored noise.

</details>


## 7. The window

**Why a window.** A record of $N$ samples is the infinite signal multiplied by a **rectangle**: 1 inside the record, 0 outside. A product in time is a convolution in frequency. So every line in the data is spread by the transform of the window.

**The two windows.**
- The rectangle: $H[n] = 1$.
- Hann: $H[n] = \sin^{2}(\pi n/N)$, which goes to zero at both ends. We divide its periodogram by $U$, the mean square of $H$, so the noise level stays right.

**The three panels.**
1. The windows in time.
2. Their transform $|W(f)|^{2}$, in decibels, against $f$ in units of $1/N$. The central lobe and the **sidelobes**: the rectangle has high sidelobes that fall slowly; Hann has a central lobe twice as wide and sidelobes far lower.
3. A strong line in white noise, near $f_{0}$. With the rectangle the sidelobes of the line cover the noise floor: this is **leakage**. With Hann the floor is visible a few bins from the line.

**Look at:** change $f_{0}$ from $0.1005$ (half-way between two frequencies of the transform, the worst case) to $0.1$ (exactly on one): the rectangle leakage almost disappears. Change $A$: the rectangle floor follows $A^{2}$, the Hann floor does not.

In [ ]:
# change these
A, f0, N, sigma_w = 100.0, 0.1005, 1000, 1.0

n = np.arange(N)
H = np.sin(np.pi * n / N) ** 2
U = np.mean(H ** 2)                       # the mean square of the Hann window

# the transform of each window, on a fine grid, in decibels, normalized to its peak
M = 64 * N                                # zero padding, only to draw the shape
bins = np.fft.fftshift(np.fft.fftfreq(M)) * N
W_rect = np.fft.fftshift(np.abs(np.fft.fft(np.ones(N), M)) ** 2)
W_hann = np.fft.fftshift(np.abs(np.fft.fft(H, M)) ** 2)
W_rect_db = 10 * np.log10(W_rect / W_rect.max() + 1e-30)
W_hann_db = 10 * np.log10(W_hann / W_hann.max() + 1e-30)

# the data: a strong line in white noise, and its periodogram with each window
rng = np.random.default_rng(4)
x = A * np.cos(2 * np.pi * f0 * n) + sigma_w * rng.normal(size=N)
f, S_rect = periodogram(x)
S_hann = periodogram(H * x)[1] / U

fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
ax[0].plot(n, np.ones(N), color=P['GREY_DARK'], lw=1.6, label='rectangle')
ax[0].plot(n, H, color=P['BLUE'], lw=1.6, label='Hann')
ax[0].set(xlabel='sample $n$', ylabel='$H[n]$', ylim=(0, 1.25), title='1. In time')
ax[0].legend(frameon=False, loc='upper center', ncol=2)
ax[1].plot(bins, W_rect_db, color=P['GREY_DARK'], lw=1.2, label='rectangle')
ax[1].plot(bins, W_hann_db, color=P['BLUE'], lw=1.2, label='Hann')
ax[1].set(xlabel='$f$ in units of $1/N$', ylabel='$|W(f)|^2$ (dB)', xlim=(-10, 10), ylim=(-100, 30),
          title='2. In frequency')
ax[1].legend(frameon=False, loc='upper center', ncol=2, fontsize=10)
ax[2].semilogy(f, S_rect, color=P['GREY_MID'], lw=1.0, label='rectangle')
ax[2].semilogy(f, S_hann, color=P['BLUE'], lw=1.0, label='Hann')
ax[2].axhline(sigma_w ** 2, color=P['BRICK'], lw=1.2, ls='--', label='the noise')
ax[2].set(xlabel='$f$ (cycles per sample)', ylabel='$\\hat{S}(f)$', xlim=(f0 - 0.05, f0 + 0.05), title='3. A line in the noise')
ax[2].legend(frameon=False, loc='upper right', fontsize=10)
plt.tight_layout()
plt.show()

### 7b. Other windows, with `scipy.signal.welch`

**The question.** Hann is not the only window. How does the estimate change with the window?

**The function.** `scipy.signal.welch` does Welch's method in one call: it cuts the record into segments of `nperseg` samples, multiplies each by the window, takes the periodogram, divides by $U$ and averages. We use no overlap, `noverlap=0`, and no detrending, `detrend=False`, as in our method by hand.

**One factor of 2.** SciPy returns a **one-sided** PSD: the power at $-f$ is added to $+f$. For a real signal it is $2\,\hat{S}(f)$ for $0 < f < 1/2$. We divide it by 2 to compare with our two-sided $\hat{S}(f)$.

**First check.** With the Hann window SciPy must give our Welch estimate by hand.

**Four windows.**
- rectangle (`'boxcar'`): the narrowest central lobe, the highest sidelobes;
- Hann (`'hann'`): sidelobes at $-31$ dB, falling fast;
- Hamming (`'hamming'`): first sidelobe lower, at $-43$ dB, but they do not fall;
- Blackman (`'blackman'`): sidelobes at $-58$ dB, the widest central lobe.

**The data.** Colored noise with $a = 0.9$, plus a strong line at $f_{0}$, half-way between two frequencies of the segment.

**Look at:** near the line, the rectangle covers the noise with leakage. Hann and Blackman show the noise floor. Hamming is in between: its first sidelobe is low, but the far sidelobes do not fall, so a strong line still leaks far away. The line is wider with Blackman: lower sidelobes cost resolution. The cell prints, far from the line, the estimate divided by the exact noise. Change $A$ (10, 100, 1000) and $D$.

In [ ]:
# change these
a, sigma_w, N, D, A = 0.9, 1.0, 65536, 1024, 300.0
f0 = 100.5 / D                              # half-way between two frequencies j/D

w, x = colored_noise(a, N, sigma_w, seed=11)
n = np.arange(N)
x = x + A * np.cos(2 * np.pi * f0 * n)
K = N // D

# 1. our Welch by hand, with Hann
n_seg = np.arange(D)
H = np.sin(np.pi * n_seg / D) ** 2
U = np.mean(H ** 2)
S_hand = np.mean([periodogram(H * seg)[1] / U for seg in x[:K * D].reshape(K, D)], axis=0)
f_hand = periodogram(np.zeros(D))[0]

# 2. SciPy, the same: Hann, no overlap, no detrending; divided by 2 to make it two-sided
f_sp, S_sp = signal.welch(x, fs=1.0, window='hann', nperseg=D, noverlap=0,
                          detrend=False, scaling='density')
S_sp_two = S_sp / 2
inside = (f_sp > 0) & (f_sp < 0.5)
print(f"Hann, largest relative difference hand - SciPy: "
      f"{np.max(np.abs(S_hand[inside] / S_sp_two[inside] - 1)):.1e}")
# SciPy's 'hann' is the periodic Hann window, sin^2(pi n / D), the one we use

# 3. four windows
windows = [('boxcar', 'rectangle', P['GREY_DARK']), ('hann', 'Hann', P['BLUE']),
           ('hamming', 'Hamming', P['SAGE_DARK']), ('blackman', 'Blackman', P['AMBER_DARK'])]
S_exact = exact_psd(f_sp, a, sigma_w)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
for win, name, c in windows:
    f_w, S_w = signal.welch(x, fs=1.0, window=win, nperseg=D, noverlap=0, detrend=False)
    S_w = S_w / 2
    far = (np.abs(f_w - f0) > 0.05) & (f_w > 0.3)
    print(f"{name:10s}: far from the line, estimate / exact = {np.mean(S_w[far] / S_exact[far]):.2f}")
    for axis in ax:
        axis.semilogy(f_w, S_w, color=c, lw=1.0, label=name)
for axis in ax:
    axis.semilogy(f_sp, S_exact, color=P['BRICK'], lw=1.6, ls='--', label='exact noise')
ax[0].set(xlabel='$f$ (cycles per sample)', ylabel='$\\hat{S}_{W}(f)$', xlim=(0, 0.5), title='the whole band')
ax[1].set(xlabel='$f$ (cycles per sample)', xlim=(f0 - 0.04, f0 + 0.04), title='near the line')
ax[1].legend(frameon=False, loc='upper right', fontsize=10)
plt.tight_layout()
plt.show()

### Exercise 4 · Find a hidden line

Make a record of $N = 4096$ samples: white noise with $\sigma_{w} = 1$ (seed 8) plus a line $0.3\cos(2\pi f_{0} n)$, with $f_{0}$ unknown to you. Use the code below to build it, without printing `f0_secret`, then compute the periodogram with the Hann window and find $f_{0}$.

```python
rng = np.random.default_rng(8)
f0_secret = 0.2 + 0.1 * rng.random()        # somewhere between 0.2 and 0.3
n = np.arange(4096)
x = 0.3 * np.cos(2 * np.pi * f0_secret * n) + rng.normal(size=4096)
```

**What you should see.** In time the line is invisible: its amplitude is a third of the noise. In frequency it is a peak about eighty times above the noise floor, between 0.2 and 0.3.

In [ ]:
# your code here

<details>
<summary>Solution</summary>

```python
rng = np.random.default_rng(8)
N = 4096
n = np.arange(N)
f0_secret = 0.2 + 0.1 * rng.random()        # somewhere between 0.2 and 0.3
x = 0.3 * np.cos(2 * np.pi * f0_secret * n) + rng.normal(size=N)
# the Hann window and its mean square, so the noise level stays right
H = np.sin(np.pi * n / N) ** 2
U = np.mean(H ** 2)
f = np.fft.rfftfreq(N)
S_hann = np.abs(np.fft.rfft(H * x)) ** 2 / (N * U)
i_peak = np.argmax(S_hann)
print(f"peak at f = {f[i_peak]:.4f}, height {S_hann[i_peak]:.1f}, median floor {np.median(S_hann):.2f}")
print(f"the secret frequency was {f0_secret:.4f}")
```
The peak is at $f = 0.2327$ cycles per sample: the secret frequency, $0.2327$, to the resolution $1/N$. Its height is $60.8$ against a median floor of $0.71$. The record has an rms of $1.04$, so in time the line, of amplitude $0.3$, cannot be seen. The periodogram collects the power of the line in one frequency, while the noise spreads over all of them.

</details>


## 8. Bias and variance

**The question.** With one record of $N$ samples, how long should the segments of Welch's method be?

**The two errors.**
- **Variance**: the scatter of the estimate around its mean. With $K = N/D$ segments it falls as $1/\sqrt{K}$. Short segments, many of them: small variance.
- **Bias**: the mean of the estimate minus the true PSD. A segment of $D$ samples cannot see details finer than $\delta f = 1/D$: it smooths the PSD over $\delta f$. Short segments: a narrow peak comes out too low and too wide.

**The noise.** The peaked noise of example 4, with a narrow peak. Its width is about $(1 - r)/\pi$ in cycles per sample.

**Look at:** the three estimates against the exact PSD. The cell prints, for each $D$, the scatter in a smooth band and the height of the peak divided by the exact one. Change $r$ (0.95, 0.99): the narrower the peak, the longer the segment it needs.

In [ ]:
# change these
f_r, r, sigma_w, N = 0.1, 0.98, 1.0, 65536
D_values = (64, 512, 4096)

# the peaked noise of example 4: two memory terms
a1 = 2 * r * np.cos(2 * np.pi * f_r)
a2 = -r ** 2
w = sigma_w * np.random.default_rng(10).normal(size=N + 1000)
x = np.zeros_like(w)
for n in range(2, len(w)):
    x[n] = a1 * x[n - 1] + a2 * x[n - 2] + w[n]
x = x[1000:]


def exact_psd2(f):
    """Return the PSD of the filter with two memory terms at the frequencies f."""
    S = sigma_w ** 2 / np.abs(1 - a1 * np.exp(-2j * np.pi * f) - a2 * np.exp(-4j * np.pi * f)) ** 2
    return S


print(f"width of the peak about (1 - r)/pi = {(1 - r) / np.pi:.4f} cycles per sample")
colors = [P['AMBER_DARK'], P['SAGE_DARK'], P['BLUE']]
f_fine = np.linspace(0, 0.5, 20001)
fig, ax = plt.subplots(2, 3, figsize=(14, 6.4), sharey='row')
for j, (D, c) in enumerate(zip(D_values, colors)):
    K = N // D
    n_seg = np.arange(D)
    H = np.sin(np.pi * n_seg / D) ** 2
    U = np.mean(H ** 2)
    # Welch: the Hann-windowed periodogram of each segment, then the average
    S_avg = np.mean([periodogram(H * seg)[1] / U for seg in x[:K * D].reshape(K, D)], axis=0)
    f = periodogram(np.zeros(D))[0]
    ratio = S_avg / exact_psd2(f)
    smooth = (f > 0.3) & (f < 0.45)
    peak_ratio = S_avg.max() / exact_psd2(f_r)
    print(f"D = {D:5d}: K = {K:4d}, resolution 1/D = {1 / D:.5f}, scatter {ratio[smooth].std():.2f}, "
          f"peak / exact peak = {peak_ratio:.2f}")
    # top row: the whole band; bottom row: the peak, enlarged
    for a in ax[:, j]:
        a.semilogy(f, S_avg, color=c, lw=1.0, label='Welch')
        a.semilogy(f_fine, exact_psd2(f_fine), color=P['BRICK'], lw=1.6, ls='--', label='exact')
    ax[0, j].set(xlim=(0, 0.5), title=f'$D$ = {D}, $K$ = {K}')
    ax[1, j].set(xlim=(f_r - 0.03, f_r + 0.03), ylim=(5, 3 * exact_psd2(f_r)), xlabel='$f$ (cycles per sample)')
ax[0, 0].set_ylabel('$\\hat{S}_{W}(f)$, whole band')
ax[1, 0].set_ylabel('$\\hat{S}_{W}(f)$, the peak')
ax[0, 2].legend(frameon=False, loc='upper right', fontsize=10)
plt.tight_layout()
plt.show()
